In [1]:
%pip install pandas sodapy requests

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached certifi-2026.10.23-py3-none-any.whl.metadata (2.5 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached idna-3.20-py3-none-any.whl (69 kB)
Using cached urllib3-2.8.0-py3-none-any.whl (135 kB)
Using cached certifi-2026.10.23-py3-none-any.whl (135 kB)

   ---------------------------------------- 0/6 [urllib3]
   ---------------------------------------- 0/6 [urllib3]
   ---------------------------------------- 0/6 [urllib3]
   ---------------------------------------- 0/6 [urllib3]
   ---------------------------------------- 0/6 [urllib3]
   ------ --------------------------------- 1/6 [idna]
   ------------- -------------------------- 2/6 [charset_normalizer]
   ------------- -------------------------- 2/6 [charset_normalizer]
   -------------------- ------------------- 


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import time
from pathlib import Path
import requests

APP_TOKEN = "GQOhdYmMh6MchEnD2t8372ECZ"
url = "https://data.cityofnewyork.us/api/v3/views/76xm-jjuj/export.csv"

query = (
    "SELECT * "
    "WHERE incident_datetime >= '2023-10-01T00:00:00' "
    "AND incident_datetime < '2026-10-01T00:00:00'"
)
output_file = Path(r"D:\DEPI-Final-Project\nyc_ems_2023_2026.csv")
output_file.parent.mkdir(parents=True, exist_ok=True)

headers = {"X-App-Token": APP_TOKEN}

start = time.time()

with requests.get(
    url, params={"query": query}, headers=headers, stream=True, timeout=(30, 300)
) as response:
  print("HTTP status:", response.status_code)
  response.raise_for_status()

  content_type = response.headers.get("Content-Type", "")
  print("Content-Type:", content_type)

  with output_file.open("wb") as f:
    for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
      if chunk:
        f.write(chunk)

print("Saved to:", output_file.resolve())
print("File size (GB):", round(output_file.stat().st_size / 1e9, 3))
print("Elapsed time (minutes):", round((time.time() - start) / 60, 2))

HTTP status: 200
Content-Type: text/csv; charset=utf-8
Saved to: D:\DEPI-Final-Project\nyc_ems_2023_2026.csv
File size (GB): 1.395
Elapsed time (minutes): 6.79


In [6]:
import csv
import time
from pathlib import Path

file_path = Path(r"D:\DEPI-Final-Project\nyc_ems_2023_2026.csv")

start = time.time()
row_count = 0

with file_path.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f)

    header = next(reader, None)

    for _ in reader:
        row_count += 1

print("Columns:", len(header) if header else 0)
print("Data rows:", f"{row_count:,}")
print("Expected rows:", f"{4_862_291:,}")
print("Matches expected:", row_count == 4_862_291)
print("File size (GB):", round(file_path.stat().st_size / 1e9, 3))
print("Verification time (minutes):", round((time.time() - start) / 60, 2))

Columns: 31
Data rows: 4,862,291
Expected rows: 4,862,291
Matches expected: True
File size (GB): 1.395
Verification time (minutes): 0.19


In [7]:
%pip install pyarrow

   ---------------------------------------- 0.0/29.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/29.8 MB ? eta -:--:--
    --------------------------------------- 0.5/29.8 MB 2.0 MB/s eta 0:00:15
   - -------------------------------------- 0.8/29.8 MB 1.7 MB/s eta 0:00:17
   - -------------------------------------- 1.3/29.8 MB 1.9 MB/s eta 0:00:16
   -- ------------------------------------- 1.8/29.8 MB 2.1 MB/s eta 0:00:14
   --- ------------------------------------ 2.6/29.8 MB 2.4 MB/s eta 0:00:12
   ---- ----------------------------------- 3.4/29.8 MB 2.6 MB/s eta 0:00:11
   ----- ---------------------------------- 4.2/29.8 MB 2.8 MB/s eta 0:00:10
   ------- -------------------------------- 5.2/29.8 MB 3.1 MB/s eta 0:00:08
   -------- ------------------------------- 6.3/29.8 MB 3.3 MB/s eta 0:00:08
   ---------- ----------------------------- 7.6/29.8 MB 3.6 MB/s eta 0:00:07
   ----------- ---------------------------- 8.9/29.8 MB 3.8 MB/s eta 0:00:06
   ----------


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd
from pathlib import Path

csv_path = Path(r"D:\DEPI-Final-Project\nyc_ems_2023_2026.csv")
parquet_path = Path(r"D:\DEPI-Final-Project\nyc_ems_2023_2026.parquet")

df = pd.read_csv(csv_path, low_memory=False)
df.to_parquet(parquet_path, index=False)

print("File size (GB):", round(parquet_path.stat().st_size / 1e9, 3))

File size (GB): 0.268
